# HERBench full → Google Drive (stream extract, không nhét 161 GB vào Colab)

Notebook **chỉ setup dataset**. Không train, không tải Ovis, không chạy eval.

Stream 17 tar part từ [DanBenAmi/HERBench](https://huggingface.co/datasets/DanBenAmi/HERBench) **thẳng vào `tar -x` trên Drive**. Không lưu `videos.tar.part.*` trên Colab hay Drive. Eval lite **không** đổi.

## Vì sao không `snapshot_download` full

Colab disk ~108 GB. Một part đã ~10 GiB; Hub còn cache blob dưới `~/.cache/huggingface` → đầy 102.9/107.7 GB trước khi Drive nhận đủ. 17 part là **một** tar bị cắt: phải `00||01||…||16` thành stdin của `tar`.

## Hardware / quota

Colab **CPU**. Drive cần **~170 GB trống** (chỉ mp4, không archive).

| Nơi | Dùng |
| --- | ---- |
| Colab `/content` | buffer vài MB + `/tmp/hf_herbench` (file nhỏ) |
| Drive | ~161 GB mp4 (335 file) |

## Layout trên Drive

```
/content/drive/MyDrive/HERBench_official/
  data/herbench_full.parquet
  data/*metadata.json
  videos/**/*.mp4                  ← 335 file (không có .part.*)
  HERBENCH_FULL_READY.json
```

## Thứ tự chạy

1. Cell 1–6 lần đầu (purge cache → parquet → stream extract).
2. Runtime đứt giữa extract: chạy lại cell 6 (`--skip-old-files`, xóa mp4 mtime mới nhất).
3. Runtime mới / chỉ kiểm tra: **cell 7**.


## 1. Configuration


In [ ]:
# 1. HERBench full → Drive config (stream extract, no GPU, no eval)
from __future__ import annotations

import os
from pathlib import Path

assert Path("/content").exists(), "Notebook này chỉ chạy trên Google Colab (/content)."

DRIVE_HERBENCH = Path("/content/drive/MyDrive/HERBench_official")
HERBENCH_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
HERBENCH_CONFIG = "full"

DO_DOWNLOAD = True            # parquet + metadata nhỏ thôi — không tải tar parts
DO_EXTRACT = True             # HTTP stream part.00–16 → tar -C Drive
DO_VERIFY_MOUNT = True
KEEP_ARCHIVES = False         # stream-only: không ghi videos.tar.part.*
USE_HF_TRANSFER = False       # hf_transfer ghi file đầy đủ xuống disk
HF_MAX_WORKERS = 2
MIN_FREE_GIB_WARN = 170.0     # Drive, cho 335 mp4
MIN_COLAB_FREE_GIB = 8.0      # cảnh báo local; không cần 161 GB trên Colab
EXPECTED_N_QUESTIONS = 27631
EXPECTED_N_MP4 = 335
EXPECTED_N_PARTS = 17
STREAM_CHUNK = 8 * 1024 * 1024

PART_SIZE_10GIB = 10737418240
EXPECTED_PART_SIZES = {
    f"videos.tar.part.{i:02d}": PART_SIZE_10GIB for i in range(17)
}
EXPECTED_PART_SIZES["videos.tar.part.03"] = 4649482240
EXPECTED_PART_SIZES["videos.tar.part.16"] = 5038397440

HERBENCH_VIDEOS = DRIVE_HERBENCH / "videos"
HERBENCH_DATA = DRIVE_HERBENCH / "data"
READY_JSON = DRIVE_HERBENCH / "HERBENCH_FULL_READY.json"
HF_TMP = Path("/tmp/hf_herbench")

os.environ["HF_HOME"] = str(HF_TMP)
os.environ["HF_HUB_CACHE"] = str(HF_TMP / "hub")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
HF_TMP.mkdir(parents=True, exist_ok=True)

print({
    "DRIVE_HERBENCH": str(DRIVE_HERBENCH),
    "HERBENCH_REVISION": HERBENCH_REVISION,
    "DO_DOWNLOAD": DO_DOWNLOAD,
    "DO_EXTRACT": DO_EXTRACT,
    "DO_VERIFY_MOUNT": DO_VERIFY_MOUNT,
    "KEEP_ARCHIVES": KEEP_ARCHIVES,
    "USE_HF_TRANSFER": USE_HF_TRANSFER,
    "HF_HOME": os.environ["HF_HOME"],
    "MIN_COLAB_FREE_GIB": MIN_COLAB_FREE_GIB,
})


## 2. Mount Drive + quota (Drive và disk Colab)

Drive cần ~170 GiB trống cho mp4. Colab local không cần 161 GB — nếu còn dưới 8 GiB, chạy cell 3 (purge cache).


In [ ]:
# 2. Mount Drive + disk quota (Drive + Colab /content)
from __future__ import annotations

import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_HERBENCH.mkdir(parents=True, exist_ok=True)
HERBENCH_VIDEOS.mkdir(parents=True, exist_ok=True)
HERBENCH_DATA.mkdir(parents=True, exist_ok=True)

drive_usage = shutil.disk_usage("/content/drive/MyDrive")
colab_usage = shutil.disk_usage("/content")
drive_free = drive_usage.free / 2**30
colab_free = colab_usage.free / 2**30
used_root = 0.0
if DRIVE_HERBENCH.exists():
    used_root = sum(p.stat().st_size for p in DRIVE_HERBENCH.rglob("*") if p.is_file()) / 2**30

print({
    "DRIVE_HERBENCH": str(DRIVE_HERBENCH),
    "drive_total_GiB": round(drive_usage.total / 2**30, 2),
    "drive_free_GiB": round(drive_free, 2),
    "colab_total_GiB": round(colab_usage.total / 2**30, 2),
    "colab_free_GiB": round(colab_free, 2),
    "herbench_used_GiB": round(used_root, 2),
})
if drive_free < MIN_FREE_GIB_WARN:
    print(
        f"WARN: Drive còn {drive_free:.1f} GiB < {MIN_FREE_GIB_WARN:.0f} GiB. "
        "Cần ~170 GiB cho 335 mp4 (không lưu tar parts)."
    )
else:
    print("Drive quota OK for extracted mp4s.")
if colab_free < MIN_COLAB_FREE_GIB:
    print(
        f"WARN: Colab còn {colab_free:.1f} GiB < {MIN_COLAB_FREE_GIB:.0f} GiB. "
        "Chạy cell 3 để xóa ~/.cache/huggingface — không tải tar lên /content."
    )
else:
    print("Colab local disk OK for streaming (no 161 GB needed).")


## 3. Purge Colab HF cache

Xóa `~/.cache/huggingface` (thường làm đầy 102.9/107.7 GB). Xóa `/content/HERBench_official` nếu lỡ snapshot vào local. Xóa leftover `videos.tar.part.*` trên Drive (notebook này không dùng archive).

**Không** xóa mp4 đã extract trên Drive.


In [ ]:
# 3. Purge Colab caches (never delete extracted Drive mp4s)
from __future__ import annotations

import shutil
from pathlib import Path

targets = [
    Path.home() / ".cache" / "huggingface",
    Path("/root/.cache/huggingface"),
    Path("/content/.cache/huggingface"),
    Path("/content/HERBench_official"),
    HF_TMP,
]
for t in targets:
    if t.is_dir():
        print("rmtree", t)
        shutil.rmtree(t, ignore_errors=True)
    elif t.is_file():
        print("unlink", t)
        t.unlink(missing_ok=True)
    else:
        print("skip missing", t)

leftover_parts = sorted(HERBENCH_VIDEOS.glob("videos.tar.part.*")) if HERBENCH_VIDEOS.exists() else []
for part in leftover_parts:
    print("unlink leftover Drive archive", part.name, part.stat().st_size)
    part.unlink(missing_ok=True)
if leftover_parts:
    print(f"Removed {len(leftover_parts)} leftover tar parts from Drive")

HF_TMP.mkdir(parents=True, exist_ok=True)
colab_free = shutil.disk_usage("/content").free / 2**30
print({"colab_free_GiB": round(colab_free, 2), "HF_TMP": str(HF_TMP)})


## 4. Dependencies

`huggingface_hub` + `datasets` + `requests`. **Không** cài `hf_transfer` (nó ghi file đầy đủ xuống disk).


In [ ]:
# 4. pip (CPU runtime, no hf_transfer)
import os
import subprocess
import sys

pkgs = ["huggingface_hub>=0.26.0", "datasets", "pandas", "pyarrow", "requests"]
subprocess.run([sys.executable, "-m", "pip", "install", "-U", *pkgs], check=True)

os.environ["HF_HOME"] = str(HF_TMP)
os.environ["HF_HUB_CACHE"] = str(HF_TMP / "hub")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
print("HF_HUB_ENABLE_HF_TRANSFER=", os.environ["HF_HUB_ENABLE_HF_TRANSFER"])
print("HF_HOME=", os.environ["HF_HOME"])


## 5. Download parquet + metadata (nhỏ)

Không tải `videos.tar.part.*`. Cache Hub nằm ở `/tmp/hf_herbench` rồi xóa sau khi copy lên Drive.


In [ ]:
# 5. snapshot small files only (no tar parts)
from __future__ import annotations

import os
import shutil

from huggingface_hub import snapshot_download

os.environ["HF_HOME"] = str(HF_TMP)
os.environ["HF_HUB_CACHE"] = str(HF_TMP / "hub")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
HF_TMP.mkdir(parents=True, exist_ok=True)

if not DO_DOWNLOAD:
    print("Skip download (DO_DOWNLOAD=False)")
else:
    print(f"Downloading metadata {HERBENCH_ID}@{HERBENCH_REVISION[:12]} → {DRIVE_HERBENCH}")
    snapshot_download(
        repo_id=HERBENCH_ID,
        repo_type="dataset",
        revision=HERBENCH_REVISION,
        local_dir=str(DRIVE_HERBENCH),
        cache_dir=str(HF_TMP / "hub"),
        max_workers=HF_MAX_WORKERS,
        allow_patterns=[
            "data/*",
            "videos/video_list.txt",
            "videos/videos_lite_info.txt",
            "videos/videos.tar.checksums.txt",
            "herbench.py",
            "README.md",
            ".gitattributes",
        ],
        ignore_patterns=[
            "videos/videos.tar.part.*",
        ],
    )
    parquet = HERBENCH_DATA / "herbench_full.parquet"
    assert parquet.is_file(), f"Thiếu {parquet}"
    print("parquet:", parquet.name, parquet.stat().st_size)
    shutil.rmtree(HF_TMP, ignore_errors=True)
    HF_TMP.mkdir(parents=True, exist_ok=True)
    print("Purged", HF_TMP, "after small snapshot")
    print("colab_free_GiB", round(shutil.disk_usage("/content").free / 2**30, 2))


## 6. Stream extract (HTTP part.00–16 → `tar -C` Drive)

Không ghi archive. Nếu đã ≥ 335 mp4 → skip. Nếu đứt giữa chừng: xóa mp4 **mtime mới nhất** (file đang ghi dở), rồi `tar --skip-old-files` và stream lại từ HF.

Giữ tab mở; extract có thể vài giờ (Drive FUSE chậm).


In [ ]:
# 6. HTTP stream videos.tar.part.00–16 into tar -x on Drive (no archive on disk)
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

import requests
from huggingface_hub import get_hf_file_metadata, hf_hub_url
from huggingface_hub.utils import build_hf_headers

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"


def list_mp4s() -> list[Path]:
    return [p for p in HERBENCH_VIDEOS.rglob("*.mp4") if p.is_file()]


def drop_truncated_newest(mp4s: list[Path]) -> None:
    empty = [p for p in mp4s if p.stat().st_size <= 0]
    for p in empty:
        print("unlink empty mp4", p)
        p.unlink(missing_ok=True)
    remain = [p for p in mp4s if p.exists() and p.stat().st_size > 0]
    if not remain:
        return
    newest = max(remain, key=lambda p: p.stat().st_mtime)
    print("unlink truncated newest mp4", newest, "bytes", newest.stat().st_size)
    newest.unlink(missing_ok=True)


def stream_part_chunks(name: str):
    rel = f"videos/{name}"
    url = hf_hub_url(HERBENCH_ID, rel, repo_type="dataset", revision=HERBENCH_REVISION)
    meta = get_hf_file_metadata(url)
    expected = EXPECTED_PART_SIZES[name]
    print(f"stream {name}: hub_size={meta.size} expected={expected}")
    headers = build_hf_headers()
    with requests.get(meta.location, stream=True, headers=headers, timeout=(60, None)) as resp:
        resp.raise_for_status()
        got = 0
        for chunk in resp.iter_content(STREAM_CHUNK):
            if not chunk:
                continue
            got += len(chunk)
            yield chunk
    if got != expected:
        raise RuntimeError(f"{name}: streamed {got} bytes, expected {expected}")
    print(f"  {name} OK {got} bytes")


if not DO_EXTRACT:
    print(f"Skip extract (DO_EXTRACT=False); mp4={len(list_mp4s())}")
else:
    for part in sorted(HERBENCH_VIDEOS.glob("videos.tar.part.*")):
        print("unlink leftover", part.name)
        part.unlink(missing_ok=True)

    mp4s = list_mp4s()
    print(f"mp4 before extract: {len(mp4s)}")
    if len(mp4s) >= EXPECTED_N_MP4:
        print(f"Already have {len(mp4s)} mp4 — skip stream")
    else:
        if mp4s:
            drop_truncated_newest(mp4s)
            print(f"mp4 after dropping truncated: {len(list_mp4s())}")
        print("Streaming HF parts 00–16 → tar --skip-old-files →", HERBENCH_VIDEOS)
        log_path = Path("/tmp/herbench_tar.log")
        with log_path.open("wb") as logf:
            tar = subprocess.Popen(
                ["tar", "-x", "--skip-old-files", "-C", str(HERBENCH_VIDEOS)],
                stdin=subprocess.PIPE,
                stdout=logf,
                stderr=logf,
            )
            assert tar.stdin is not None
            try:
                names = [f"videos.tar.part.{i:02d}" for i in range(EXPECTED_N_PARTS)]
                for name in names:
                    print("colab_free_GiB", round(shutil.disk_usage("/content").free / 2**30, 2))
                    for chunk in stream_part_chunks(name):
                        tar.stdin.write(chunk)
                    tar.stdin.flush()
                    print(name, "piped into tar")
            except Exception:
                tar.kill()
                raise
            finally:
                try:
                    tar.stdin.close()
                except Exception:
                    pass
            rc = tar.wait()
        print("tar log tail:")
        try:
            print("\n".join(log_path.read_text(errors="replace").splitlines()[-20:]))
        except Exception as exc:
            print("cannot read tar log", exc)
        if rc != 0:
            raise RuntimeError(f"tar exited {rc}. Chạy lại cell 6 (resume skip-old-files).")
        print("Extract stream finished")

    n_mp4 = len(list_mp4s())
    print({"videos_mp4": n_mp4, "expected": EXPECTED_N_MP4})
    leftover = list(HERBENCH_VIDEOS.glob("videos.tar.part.*"))
    for part in leftover:
        part.unlink(missing_ok=True)
    if n_mp4 < EXPECTED_N_MP4:
        raise RuntimeError(
            f"Chỉ có {n_mp4} mp4 (need {EXPECTED_N_MP4}). "
            "Chạy lại cell 6 — stream lại từ HF, skip file đã đủ."
        )
    print("Extract OK — no tar parts stored")


## 7. Test mount (runtime mới, không download)

Cell **độc lập**: `force_remount`, đọc parquet local (fallback Hub), resolve `video_path`, probe 3 mp4, ghi `HERBENCH_FULL_READY.json`.

Kỳ vọng: **27,631** câu, **335** mp4, **0** missing video. Không cần tar parts trên Drive.


In [ ]:
# 7. Independent Drive mount test + HERBENCH_FULL_READY.json
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

try:
    _run_verify = bool(DO_VERIFY_MOUNT)
except NameError:
    _run_verify = True

DRIVE_HERBENCH = Path("/content/drive/MyDrive/HERBench_official")
HERBENCH_VIDEOS = DRIVE_HERBENCH / "videos"
HERBENCH_DATA = DRIVE_HERBENCH / "data"
READY_JSON = DRIVE_HERBENCH / "HERBENCH_FULL_READY.json"
HERBENCH_ID = "DanBenAmi/HERBench"
HERBENCH_REVISION = "30b42e97180f2c7c0fdc3f2aed412ae04f0366ca"
EXPECTED_N_QUESTIONS = 27631
EXPECTED_N_MP4 = 335


def resolve_video(
    annotation_path: str,
    herbench_root: Path,
    videos_root: Path,
    name_index: dict[str, list[Path]],
) -> Path | None:
    rel = Path(str(annotation_path))
    candidates = [
        herbench_root / rel,
        videos_root / rel,
        videos_root / rel.name,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    matches = name_index.get(rel.name, [])
    return matches[0].resolve() if len(matches) == 1 else None


def probe_mp4(path: Path) -> dict:
    info = {"path": str(path), "bytes": path.stat().st_size, "ok": False, "n_frames": None, "error": None}
    if info["bytes"] <= 0:
        info["error"] = "empty file"
        return info
    try:
        import cv2

        cap = cv2.VideoCapture(str(path))
        ok_open = bool(cap.isOpened())
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0) if ok_open else 0
        cap.release()
        info["ok"] = ok_open and info["bytes"] > 0
        info["n_frames"] = n
        if not ok_open:
            info["error"] = "cv2 failed to open"
        return info
    except Exception as exc:
        info["error"] = f"cv2: {exc}"
    try:
        import decord

        vr = decord.VideoReader(str(path))
        info["ok"] = True
        info["n_frames"] = len(vr)
        info["error"] = None
        return info
    except Exception as exc:
        info["error"] = f"{info['error']}; decord: {exc}"
        info["ok"] = info["bytes"] > 0
        return info


def run_mount_test() -> None:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)

    assert DRIVE_HERBENCH.is_dir(), f"Chưa có {DRIVE_HERBENCH} — chạy cell 1–6 trước."
    assert HERBENCH_VIDEOS.is_dir(), f"Chưa có {HERBENCH_VIDEOS}"

    leftover_parts = list(HERBENCH_VIDEOS.glob("videos.tar.part.*"))
    if leftover_parts:
        print("WARN: leftover tar parts on Drive (stream mode không cần):", [p.name for p in leftover_parts])

    mp4s = sorted(p for p in HERBENCH_VIDEOS.rglob("*.mp4") if p.is_file())
    n_mp4 = len(mp4s)
    name_index: dict[str, list[Path]] = {}
    for p in mp4s:
        name_index.setdefault(p.name, []).append(p)
    print("n_mp4:", n_mp4, "expected:", EXPECTED_N_MP4)
    assert n_mp4 >= 1, f"Không thấy mp4 dưới {HERBENCH_VIDEOS} — chạy stream extract cell 6."

    parquet = HERBENCH_DATA / "herbench_full.parquet"
    if parquet.is_file():
        import pandas as pd

        df = pd.read_parquet(parquet)
        rows = df.to_dict(orient="records")
        source = f"local:{parquet}"
    else:
        from datasets import load_dataset

        ds = load_dataset(HERBENCH_ID, "full", split="test", revision=HERBENCH_REVISION)
        rows = [dict(item) for item in ds]
        source = f"hub:{HERBENCH_ID}/full@{HERBENCH_REVISION[:12]}"

    n_questions = len(rows)
    video_ids = {str(r.get("video_id", "")) for r in rows}
    unique_paths = sorted({str(r.get("video_path", "") or "") for r in rows})
    resolved_map = {
        vp: resolve_video(vp, DRIVE_HERBENCH, HERBENCH_VIDEOS, name_index) for vp in unique_paths
    }
    missing = []
    resolved_ok = 0
    for row in rows:
        vp = str(row.get("video_path", "") or "")
        if resolved_map.get(vp) is None:
            missing.append({"question_id": str(row.get("question_id", "")), "video_path": vp})
        else:
            resolved_ok += 1

    print(
        {
            "source": source,
            "n_questions": n_questions,
            "unique_video_id": len(video_ids),
            "unique_video_path": len(unique_paths),
            "n_mp4": n_mp4,
            "resolved_ok": resolved_ok,
            "missing": len(missing),
            "missing_head": missing[:5],
        }
    )

    if n_questions != EXPECTED_N_QUESTIONS:
        print(f"WARN: n_questions={n_questions} expected={EXPECTED_N_QUESTIONS}")
    if n_mp4 < EXPECTED_N_MP4:
        print(f"WARN: n_mp4={n_mp4} < {EXPECTED_N_MP4}")
    if missing:
        print(f"WARN: {len(missing)} questions missing videos")

    probe_targets = [mp4s[0], mp4s[len(mp4s) // 2], mp4s[-1]] if mp4s else []
    probes = [probe_mp4(p) for p in probe_targets]
    for item in probes:
        print("probe", item)
    assert any(item["ok"] for item in probes), "Không mở được mp4 mẫu (cv2/decord)."

    ready = {
        "ok": n_mp4 >= EXPECTED_N_MP4 and len(missing) == 0 and n_questions == EXPECTED_N_QUESTIONS,
        "path": str(DRIVE_HERBENCH),
        "videos_root": str(HERBENCH_VIDEOS),
        "revision": HERBENCH_REVISION,
        "config": "full",
        "n_mp4": n_mp4,
        "n_questions": n_questions,
        "unique_video_id": len(video_ids),
        "missing_videos": len(missing),
        "source": source,
        "probes": probes,
        "eval_hint": {
            "HERBENCH_ROOT": str(DRIVE_HERBENCH),
            "HERBENCH_VIDEOS": str(HERBENCH_VIDEOS),
            "HERBENCH_CONFIG": "full",
        },
        "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    READY_JSON.write_text(json.dumps(ready, indent=2))
    print("Wrote", READY_JSON)
    print(
        json.dumps(
            {k: ready[k] for k in ("ok", "path", "n_mp4", "n_questions", "missing_videos")},
            indent=2,
        )
    )

    assert n_mp4 >= EXPECTED_N_MP4, f"mp4={n_mp4} < {EXPECTED_N_MP4}"
    assert n_questions == EXPECTED_N_QUESTIONS, f"questions={n_questions} != {EXPECTED_N_QUESTIONS}"
    assert not missing, f"{len(missing)} questions missing videos; first={missing[:5]}"
    print("Mount test OK — eval full có thể dùng", DRIVE_HERBENCH)


if not _run_verify:
    print("Skip verify (DO_VERIFY_MOUNT=False)")
else:
    run_mount_test()
